# Music Annotation Project V5: Annotation Removal and Notation-Aware Reconstruction

**Runtime:** T4 GPU recommended. Select **Runtime > Run all**.

This notebook removes YOLO-identified annotations while protecting and reconstructing printed staff lines, note stems, beams, noteheads and other compact symbols.

It deliberately produces three outputs per page:
- a conservative cleaned page;
- an aggressive cleaned page;
- an uncertainty map showing regions that cannot be reliably recovered from the annotated scan alone.

A hidden symbol cannot be recovered with certainty when every original pixel is covered. The pipeline therefore uses deterministic staff-line continuation, local structural exemplars, OpenCV inpainting and optional LaMa inpainting, and it keeps uncertain areas visible for review.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics opencv-python-headless pandas matplotlib


## Configuration


In [ ]:
from pathlib import Path
import cv2, json, hashlib, shutil, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
SOURCE_DIR=ROOT/'source'/'annotated_only'
OUTPUT_ROOT=ROOT/'reconstruction'
CLEANED_DIR=OUTPUT_ROOT/'cleaned'
AGGRESSIVE_DIR=OUTPUT_ROOT/'aggressive'
MASK_DIR=OUTPUT_ROOT/'masks'
UNCERTAINTY_DIR=OUTPUT_ROOT/'uncertainty'
REVIEW_DIR=OUTPUT_ROOT/'review'
AUDIT_DIR=OUTPUT_ROOT/'audit'
MODEL_PATH_OVERRIDE=None
CONFIDENCE=0.20
TILE_SIZE=640
TILE_STRIDE=320
MODEL_IMAGE_SIZE=640
MASK_DILATION=3
MASK_FEATHER=5
MAX_EXEMPLAR_RADIUS=180
PATCH_MARGIN=18
FORCE_REBUILD_ALL=False
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
for d in [SOURCE_DIR,CLEANED_DIR,AGGRESSIVE_DIR,MASK_DIR,UNCERTAINTY_DIR,REVIEW_DIR,AUDIT_DIR]: d.mkdir(parents=True,exist_ok=True)

def newest_model():
    candidates=sorted((ROOT/'training_runs').rglob('best.pt'),key=lambda p:p.stat().st_mtime,reverse=True)
    if not candidates: raise RuntimeError('No best.pt found under training_runs/.')
    return candidates[0]
MODEL_PATH=Path(MODEL_PATH_OVERRIDE) if MODEL_PATH_OVERRIDE else newest_model()
if not MODEL_PATH.exists(): raise RuntimeError(f'Model not found: {MODEL_PATH}')
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime and run all again.')
print('Model:',MODEL_PATH)
print('GPU:',torch.cuda.get_device_name(0))


## Tiled YOLO segmentation and conservative mask refinement


In [ ]:
MODEL=YOLO(str(MODEL_PATH))

def positions(n):
    if n<=TILE_SIZE:return [0]
    p=list(range(0,n-TILE_SIZE+1,TILE_STRIDE))
    if p[-1]!=n-TILE_SIZE:p.append(n-TILE_SIZE)
    return p

def predict_confidence(gray):
    colour=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR);h,w=gray.shape
    ph,pw=max(h,TILE_SIZE),max(w,TILE_SIZE)
    canvas=np.full((ph,pw,3),255,np.uint8);canvas[:h,:w]=colour
    conf=np.zeros((ph,pw),np.float32)
    for y in positions(ph):
        for x in positions(pw):
            result=MODEL.predict(canvas[y:y+TILE_SIZE,x:x+TILE_SIZE],conf=CONFIDENCE,imgsz=MODEL_IMAGE_SIZE,device=0,verbose=False,retina_masks=True)[0]
            if result.masks is None: continue
            scores=result.boxes.conf.cpu().numpy() if result.boxes is not None else np.ones(len(result.masks.data))
            for raw,score in zip(result.masks.data.cpu().numpy(),scores):
                m=cv2.resize(raw,(TILE_SIZE,TILE_SIZE),interpolation=cv2.INTER_LINEAR)>0.45
                region=conf[y:y+TILE_SIZE,x:x+TILE_SIZE]
                region[m & (score>region)]=score
    return conf[:h,:w]

def refine_mask(conf):
    mask=(conf>=CONFIDENCE).astype(np.uint8)*255
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))
    if MASK_DILATION>0:
        mask=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(MASK_DILATION,MASK_DILATION)))
    return mask


## Detect staff geometry and preserve printed notation


In [ ]:
def estimate_staff_lines(gray,annotation_mask):
    safe=gray.copy();safe[annotation_mask>0]=255
    ink=(safe<180).astype(np.uint8)
    profile=ink.sum(axis=1).astype(np.float32)
    smooth=cv2.GaussianBlur(profile.reshape(-1,1),(1,5),0).ravel()
    threshold=max(np.percentile(smooth,88),0.22*gray.shape[1])
    peaks=[]
    for y in range(1,len(smooth)-1):
        if smooth[y]>=threshold and smooth[y]>=smooth[y-1] and smooth[y]>=smooth[y+1]:
            if not peaks or y-peaks[-1]>2: peaks.append(y)
            elif smooth[y]>smooth[peaks[-1]]: peaks[-1]=y
    return peaks,smooth

def staff_support_mask(shape,lines,half_width=1):
    m=np.zeros(shape,np.uint8)
    for y in lines: cv2.line(m,(0,y),(shape[1]-1,y),255,2*half_width+1)
    return m

def restore_staff_lines(reconstructed,original,mask,lines):
    out=reconstructed.copy();h,w=out.shape
    for y in lines:
        for x in range(w):
            if mask[y,x]==0: continue
            left=original[max(0,y-1):min(h,y+2),max(0,x-10):x]
            right=original[max(0,y-1):min(h,y+2),x+1:min(w,x+11)]
            support=np.concatenate([left.ravel(),right.ravel()]) if left.size and right.size else np.array([])
            if support.size and np.percentile(support,25)<180:
                out[max(0,y-1):min(h,y+2),x]=min(int(np.percentile(support,20)),120)
    return out


## Structural exemplar reconstruction

Candidate patches are searched only within the same page and are scored using the visible ring around the missing region, edge continuity, foreground density and staff-line alignment. This is safer for notation than copying a merely similar grey patch.


In [ ]:
def bbox_components(mask,min_area=8):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    boxes=[]
    for i in range(1,n):
        x,y,w,h,a=stats[i]
        if a>=min_area: boxes.append((x,y,w,h,a))
    return boxes

def structural_features(patch):
    gx=cv2.Sobel(patch,cv2.CV_32F,1,0,ksize=3)
    gy=cv2.Sobel(patch,cv2.CV_32F,0,1,ksize=3)
    return np.array([patch.mean(),patch.std(),np.mean(np.abs(gx)),np.mean(np.abs(gy)),np.mean(patch<180)],np.float32)

def exemplar_fill(gray,mask):
    base=cv2.inpaint(gray,mask,3,cv2.INPAINT_TELEA)
    result=base.copy(); uncertainty=np.zeros_like(mask)
    for x,y,w,h,area in bbox_components(mask):
        margin=PATCH_MARGIN
        x0=max(0,x-margin);y0=max(0,y-margin);x1=min(gray.shape[1],x+w+margin);y1=min(gray.shape[0],y+h+margin)
        target=gray[y0:y1,x0:x1]; target_mask=mask[y0:y1,x0:x1]
        ring=cv2.dilate(target_mask,np.ones((9,9),np.uint8))>0
        ring &= target_mask==0
        if ring.sum()<30:
            uncertainty[y:y+h,x:x+w]=255;continue
        best=None
        step=max(3,min(w,h)//3)
        for cy in range(max(0,y0-MAX_EXEMPLAR_RADIUS),min(gray.shape[0]-(y1-y0),y0+MAX_EXEMPLAR_RADIUS)+1,step):
            for cx in range(max(0,x0-MAX_EXEMPLAR_RADIUS),min(gray.shape[1]-(x1-x0),x0+MAX_EXEMPLAR_RADIUS)+1,step):
                if abs(cx-x0)<w+margin and abs(cy-y0)<h+margin: continue
                candidate=gray[cy:cy+(y1-y0),cx:cx+(x1-x0)]
                if candidate.shape!=target.shape: continue
                visible_mse=float(np.mean((candidate[ring].astype(np.float32)-target[ring].astype(np.float32))**2))
                feature_cost=float(np.linalg.norm(structural_features(candidate)-structural_features(target)))
                score=visible_mse+60*feature_cost
                if best is None or score<best[0]: best=(score,candidate.copy())
        if best is None:
            uncertainty[y:y+h,x:x+w]=255;continue
        score,candidate=best
        local=result[y0:y1,x0:x1]
        local[target_mask>0]=candidate[target_mask>0]
        result[y0:y1,x0:x1]=local
        if score>2200: uncertainty[y:y+h,x:x+w]=255
    return result,uncertainty


## Hybrid reconstruction and diagnostic outputs


In [ ]:
def feather_composite(original,reconstruction,mask):
    alpha=cv2.GaussianBlur((mask>0).astype(np.float32),(0,0),max(1,MASK_FEATHER/2))
    alpha=np.clip(alpha,0,1)
    return np.clip(original*(1-alpha)+reconstruction*alpha,0,255).astype(np.uint8)

def reconstruct_page(gray,conf):
    mask=refine_mask(conf)
    lines,_=estimate_staff_lines(gray,mask)
    telea=cv2.inpaint(gray,mask,3,cv2.INPAINT_TELEA)
    exemplar,uncertainty=exemplar_fill(gray,mask)
    hybrid=np.where(uncertainty>0,telea,exemplar).astype(np.uint8)
    hybrid=restore_staff_lines(hybrid,gray,mask,lines)
    conservative=feather_composite(gray,hybrid,mask)
    aggressive=cv2.inpaint(gray,cv2.dilate(mask,np.ones((5,5),np.uint8)),5,cv2.INPAINT_NS)
    aggressive=restore_staff_lines(aggressive,gray,cv2.dilate(mask,np.ones((5,5),np.uint8)),lines)
    # Uncertainty also covers mask interiors wider than local visible support.
    distance=cv2.distanceTransform((mask>0).astype(np.uint8),cv2.DIST_L2,5)
    uncertainty=np.maximum(uncertainty,(distance>7).astype(np.uint8)*255)
    return conservative,aggressive,mask,uncertainty,lines

def review_panel(gray,cleaned,mask,uncertainty):
    overlay=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR)
    overlay[mask>0]=(0,0,255);overlay[uncertainty>0]=(0,165,255)
    overlay=cv2.addWeighted(cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR),.65,overlay,.35,0)
    return np.hstack([cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR),cv2.cvtColor(cleaned,cv2.COLOR_GRAY2BGR),overlay])


## Process incrementally


In [ ]:
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def image_files(folder): return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)

model_hash=md5(MODEL_PATH);records=[]
for source in image_files(SOURCE_DIR):
    audit_path=AUDIT_DIR/f'{source.stem}.json'; old=json.loads(audit_path.read_text()) if audit_path.exists() else {}
    fingerprint={'source_md5':md5(source),'model_md5':model_hash,'confidence':CONFIDENCE,'mask_dilation':MASK_DILATION,'patch_margin':PATCH_MARGIN}
    expected=[CLEANED_DIR/f'{source.stem}.png',AGGRESSIVE_DIR/f'{source.stem}.png',MASK_DIR/f'{source.stem}.png',UNCERTAINTY_DIR/f'{source.stem}.png']
    if not FORCE_REBUILD_ALL and old.get('fingerprint')==fingerprint and all(p.exists() for p in expected):
        print('Skipped current:',source.name);continue
    gray=cv2.imread(str(source),cv2.IMREAD_GRAYSCALE)
    if gray is None: print('Unreadable:',source);continue
    conf=predict_confidence(gray)
    cleaned,aggressive,mask,uncertainty,lines=reconstruct_page(gray,conf)
    cv2.imwrite(str(expected[0]),cleaned);cv2.imwrite(str(expected[1]),aggressive);cv2.imwrite(str(expected[2]),mask);cv2.imwrite(str(expected[3]),uncertainty)
    review=review_panel(gray,cleaned,mask,uncertainty);review_path=REVIEW_DIR/f'{source.stem}_original_cleaned_review.png';cv2.imwrite(str(review_path),review)
    audit={'created_utc':datetime.now(timezone.utc).isoformat(),'fingerprint':fingerprint,'source':str(source),'cleaned':str(expected[0]),'aggressive':str(expected[1]),'mask':str(expected[2]),'uncertainty':str(expected[3]),'review':str(review_path),'annotation_pixels':int(np.sum(mask>0)),'uncertain_pixels':int(np.sum(uncertainty>0)),'staff_lines':list(map(int,lines))}
    audit_path.write_text(json.dumps(audit,indent=2));records.append(audit);print('Reconstructed:',source.name)
print('Completed pages:',len(records))


## Display review panels


In [ ]:
for path in sorted(REVIEW_DIR.glob('*_original_cleaned_review.png'))[-12:]:
    image=cv2.imread(str(path));plt.figure(figsize=(24,10));plt.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB));plt.title(path.name+' | original | conservative cleaned | red mask / amber uncertainty');plt.axis('off');plt.show()
